# LIBERO Baseline Smoke Checks (Step 1, Local Mac, No Repo Edits)

This notebook runs a strict local **smoke validation** of LIBERO's original baseline setup using default activations and no repository code changes.

What this verifies:
1. The active interpreter and core imports are healthy.
2. `robosuite` imports when using a runtime-only workaround (`NUMBA_DISABLE_JIT=1`).
3. `LIBERO_OBJECT` dataset files exist and load.
4. `BCTransformerPolicy` computes a finite loss on real data.
5. Full local training failures are reproducible and understood.

What this does **not** do:
- It does not complete full local training.
- It does not change activation functions.
- It does not submit cluster jobs.


## Why these checks matter

LIBERO training touches a lot of moving parts (Hydra config, robosuite, datasets, model code, multiprocessing).  
These smoke checks isolate each layer so we know exactly where local execution is valid and where it fails.


In [23]:
# Step 1: Use one interpreter only.
# Everything in this notebook will run with THIS python executable.

import os
import sys
import subprocess
from pathlib import Path

REPO_ROOT = Path('/Users/evanmanolis/Desktop/LIBERO_FORK').resolve()
PYTHON_BIN = Path(sys.executable).resolve()

print('Python executable:', PYTHON_BIN)
print('Python version   :', sys.version)
print('Repo root exists :', REPO_ROOT.exists())

# Optional strict check: this notebook should run from your 'libero' env.
assert 'miniforge' in str(PYTHON_BIN), 'Unexpected python path; verify kernel selection.'


Python executable: /opt/homebrew/Caskroom/miniforge/base/envs/libero/bin/python3.8
Python version   : 3.8.13 | packaged by conda-forge | (default, Mar 25 2022, 06:05:16) 
[Clang 12.0.1 ]
Repo root exists : True


In [24]:
# Step 2: Verify core dependencies from this exact interpreter.

import hydra
import yaml
import imageio
import h5py
import robomimic
import torch

print('core imports ok')
print('torch version:', torch.__version__)


core imports ok
torch version: 1.11.0


In [25]:
# Utility helper for running subprocess commands with clear output.

def run_cmd(cmd, env=None, cwd=REPO_ROOT, expect_success=True, show_output=True):
    merged_env = os.environ.copy()
    if env:
        merged_env.update(env)

    print('\n$ ' + ' '.join(cmd))
    result = subprocess.run(
        cmd,
        cwd=str(cwd),
        env=merged_env,
        capture_output=True,
        text=True,
    )

    if show_output:
        if result.stdout:
            print(result.stdout)
        if result.stderr:
            print(result.stderr)

    print('exit code:', result.returncode)

    if expect_success and result.returncode != 0:
        raise RuntimeError('Command failed unexpectedly.')

    return result


In [26]:
# Step 3: Apply runtime-only robosuite workaround (no repo edits).
# We set NUMBA_DISABLE_JIT=1 ONLY for this command environment.

_ = run_cmd(
    [str(PYTHON_BIN), '-c', "import robosuite; print('robosuite version:', robosuite.__version__)"],
    env={'NUMBA_DISABLE_JIT': '1'},
    expect_success=True,
)



$ /opt/homebrew/Caskroom/miniforge/base/envs/libero/bin/python3.8 -c import robosuite; print('robosuite version:', robosuite.__version__)
robosuite version: 1.4.0

exit code: 0


In [27]:
# Step 4: Confirm dataset files exist and are readable.

dataset_dir = Path('/Users/evanmanolis/LIBERO/libero/datasets/libero_object')
demo_files = sorted(dataset_dir.glob('*_demo.hdf5'))

print('Dataset dir:', dataset_dir)
print('Exists     :', dataset_dir.exists())
print('Demo count :', len(demo_files))
for f in demo_files:
    print(' -', f.name)

assert dataset_dir.exists(), 'Dataset directory is missing.'
assert len(demo_files) == 10, f'Expected 10 demo files, found {len(demo_files)}'


Dataset dir: /Users/evanmanolis/LIBERO/libero/datasets/libero_object
Exists     : True
Demo count : 10
 - pick_up_the_alphabet_soup_and_place_it_in_the_basket_demo.hdf5
 - pick_up_the_bbq_sauce_and_place_it_in_the_basket_demo.hdf5
 - pick_up_the_butter_and_place_it_in_the_basket_demo.hdf5
 - pick_up_the_chocolate_pudding_and_place_it_in_the_basket_demo.hdf5
 - pick_up_the_cream_cheese_and_place_it_in_the_basket_demo.hdf5
 - pick_up_the_ketchup_and_place_it_in_the_basket_demo.hdf5
 - pick_up_the_milk_and_place_it_in_the_basket_demo.hdf5
 - pick_up_the_orange_juice_and_place_it_in_the_basket_demo.hdf5
 - pick_up_the_salad_dressing_and_place_it_in_the_basket_demo.hdf5
 - pick_up_the_tomato_sauce_and_place_it_in_the_basket_demo.hdf5


## Step 5: Benchmark + dataset loading smoke

This section mirrors the non-training data path from `libero/lifelong/main.py`:
- Compose default config via Hydra.
- Override benchmark + data paths.
- Instantiate `LIBERO_OBJECT`.
- Load all 10 task datasets.


In [28]:
# Step 5: Load LIBERO_OBJECT benchmark and all 10 datasets (no training loop).

import os
from hydra import compose, initialize_config_dir
from libero.libero.benchmark import get_benchmark
from libero.lifelong.datasets import get_dataset

config_dir = os.path.join(str(REPO_ROOT), 'libero', 'configs')

with initialize_config_dir(config_dir=config_dir, version_base=None):
    cfg = compose(
        config_name='config',
        overrides=[
            'benchmark_name=LIBERO_OBJECT',
            'folder=/Users/evanmanolis/LIBERO/libero/datasets',
            'bddl_folder=/Users/evanmanolis/Desktop/LIBERO_FORK/libero/libero/bddl_files',
            'init_states_folder=/Users/evanmanolis/Desktop/LIBERO_FORK/libero/libero/init_files',
        ],
    )

benchmark = get_benchmark(cfg.benchmark_name)(cfg.data.task_order_index)
print('benchmark:', benchmark.name)
print('n_tasks  :', benchmark.n_tasks)

for i in range(benchmark.n_tasks):
    dataset_path = os.path.join(cfg.folder, benchmark.get_task_demonstration(i))
    ds, shape_meta = get_dataset(
        dataset_path=dataset_path,
        obs_modality=cfg.data.obs.modality,
        initialize_obs_utils=(i == 0),
        seq_len=cfg.data.seq_len,
    )
    print(f'[{i}] loaded -> {dataset_path}')

print('DATASET LOADING SMOKE PASS')


[info] using task orders [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
benchmark: libero_object
n_tasks  : 10

============= Initialized Observation Utils with Obs Spec =============

using obs modality: rgb with keys: ['eye_in_hand_rgb', 'agentview_rgb']
using obs modality: depth with keys: []
using obs modality: low_dim with keys: ['joint_states', 'gripper_states']
SequenceDataset: loading dataset into memory...
100%|██████████| 50/50 [00:00<00:00, 3660.85it/s]
[0] loaded -> /Users/evanmanolis/LIBERO/libero/datasets/libero_object/pick_up_the_alphabet_soup_and_place_it_in_the_basket_demo.hdf5
SequenceDataset: loading dataset into memory...
100%|██████████| 50/50 [00:00<00:00, 1373.62it/s]
[1] loaded -> /Users/evanmanolis/LIBERO/libero/datasets/libero_object/pick_up_the_cream_cheese_and_place_it_in_the_basket_demo.hdf5
SequenceDataset: loading dataset into memory...
100%|██████████| 50/50 [00:00<00:00, 742.58it/s]
[2] loaded -> /Users/evanmanolis/LIBERO/libero/datasets/libero_object/pick_up_the_salad

## Step 6: One-batch default policy loss smoke

This confirms model-side baseline logic works end-to-end on real batch data:
- Use default `bc_transformer_policy` config.
- Build task embeddings.
- Build `SequenceVLDataset` for task 0.
- Pull one batch with `DataLoader(num_workers=0)`.
- Compute `policy.compute_loss(batch)` and verify finite output.


In [29]:
# Step 6: One-batch default-policy forward/loss smoke on real data.
# Important: we convert Hydra config to EasyDict exactly like main.py does.

import yaml
from easydict import EasyDict
from omegaconf import OmegaConf
from torch.utils.data import DataLoader

from libero.lifelong.datasets import SequenceVLDataset
from libero.lifelong.models.base_policy import get_policy_class
from libero.lifelong.utils import get_task_embs

# Re-compose config, then mirror main.py conversion path.
with initialize_config_dir(config_dir=config_dir, version_base=None):
    hydra_cfg = compose(
        config_name='config',
        overrides=[
            'benchmark_name=LIBERO_OBJECT',
            'policy=bc_transformer_policy',
            'device=cpu',
            'folder=/Users/evanmanolis/LIBERO/libero/datasets',
            'bddl_folder=/Users/evanmanolis/Desktop/LIBERO_FORK/libero/libero/bddl_files',
            'init_states_folder=/Users/evanmanolis/Desktop/LIBERO_FORK/libero/libero/init_files',
        ],
    )

cfg = EasyDict(yaml.safe_load(OmegaConf.to_yaml(hydra_cfg)))
benchmark = get_benchmark(cfg.benchmark_name)(cfg.data.task_order_index)

# Build task-0 dataset and shape metadata.
path0 = os.path.join(cfg.folder, benchmark.get_task_demonstration(0))
ds0, shape_meta = get_dataset(
    dataset_path=path0,
    obs_modality=cfg.data.obs.modality,
    initialize_obs_utils=True,
    seq_len=cfg.data.seq_len,
)

# Build real task embeddings from descriptions (default baseline behavior).
all_desc = [benchmark.get_task(i).language for i in range(benchmark.n_tasks)]
task_embs = get_task_embs(cfg, all_desc)
benchmark.set_task_embs(task_embs)

# Wrap into vision-language dataset and fetch a single batch.
vl_ds0 = SequenceVLDataset(ds0, task_embs[0])
loader = DataLoader(vl_ds0, batch_size=2, shuffle=False, num_workers=0)
batch = next(iter(loader))

# Build default policy class and compute one loss value.
policy = get_policy_class(cfg.policy.policy_type)(cfg, shape_meta)
policy = policy.to('cpu')
policy.eval()

with torch.no_grad():
    loss = policy.compute_loss(batch)

print('task0 dataset:', path0)
print('loss:', float(loss))
print('isfinite:', bool(torch.isfinite(loss).item()))
assert bool(torch.isfinite(loss).item()), 'Loss is not finite.'
print('ONE-BATCH POLICY LOSS SMOKE PASS')


[info] using task orders [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]

============= Initialized Observation Utils with Obs Spec =============

using obs modality: rgb with keys: ['eye_in_hand_rgb', 'agentview_rgb']
using obs modality: depth with keys: []
using obs modality: low_dim with keys: ['joint_states', 'gripper_states']
SequenceDataset: loading dataset into memory...
100%|██████████| 50/50 [00:00<00:00, 4872.90it/s]


/opt/homebrew/Caskroom/miniforge/base/envs/libero/lib/python3.8/site-packages/torch/functional.py:568: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at  /Users/distiller/project/pytorch/aten/src/ATen/native/TensorShape.cpp:2228.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]


task0 dataset: /Users/evanmanolis/LIBERO/libero/datasets/libero_object/pick_up_the_alphabet_soup_and_place_it_in_the_basket_demo.hdf5
loss: 5.616585731506348
isfinite: True
ONE-BATCH POLICY LOSS SMOKE PASS


## Step 7: Reproduce expected local full-training blockers

We intentionally run `libero/lifelong/main.py` in two known-failure modes and assert the exact failure reason.

This documents the local boundary clearly:
1. `num_workers=0` -> `persistent_workers option needs num_workers > 0`
2. `num_workers=1` -> `h5py objects cannot be pickled`


In [30]:
# Step 7a: Reproduce blocker #1 (expected fail):
# train.num_workers=0 -> persistent_workers option needs num_workers > 0

cmd_workers0 = [
    str(PYTHON_BIN),
    'libero/lifelong/main.py',
    'benchmark_name=LIBERO_OBJECT',
    'policy=bc_transformer_policy',
    'lifelong=base',
    'seed=10000',
    'folder=/Users/evanmanolis/LIBERO/libero/datasets',
    'bddl_folder=/Users/evanmanolis/Desktop/LIBERO_FORK/libero/libero/bddl_files',
    'init_states_folder=/Users/evanmanolis/Desktop/LIBERO_FORK/libero/libero/init_files',
    'device=cpu',
    'train.n_epochs=0',
    'eval.eval=false',
    'train.num_workers=0',
    'eval.num_workers=0',
]

res0 = run_cmd(cmd_workers0, env={'NUMBA_DISABLE_JIT': '1'}, expect_success=False)
assert res0.returncode != 0, 'Expected failure did not occur.'
assert 'persistent_workers option needs num_workers > 0' in (res0.stdout + res0.stderr)
print('EXPECTED BLOCKER #1 CONFIRMED')



$ /opt/homebrew/Caskroom/miniforge/base/envs/libero/bin/python3.8 libero/lifelong/main.py benchmark_name=LIBERO_OBJECT policy=bc_transformer_policy lifelong=base seed=10000 folder=/Users/evanmanolis/LIBERO/libero/datasets bddl_folder=/Users/evanmanolis/Desktop/LIBERO_FORK/libero/libero/bddl_files init_states_folder=/Users/evanmanolis/Desktop/LIBERO_FORK/libero/libero/init_files device=cpu train.n_epochs=0 eval.eval=false train.num_workers=0 eval.num_workers=0
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
{ 'bddl_folder': '/Users/evanmanolis/Desktop/LIBERO_FORK/libero/libero/bddl_files',
  'benchmark_name': 'LIBERO_OBJECT',
  'data': { 'action_scale': 1.0,
            'affine_translate': 4,
            'data_moda

In [31]:
# Step 7b: Reproduce blocker #2 (expected fail):
# train.num_workers=1 -> h5py objects cannot be pickled

cmd_workers1 = [
    str(PYTHON_BIN),
    'libero/lifelong/main.py',
    'benchmark_name=LIBERO_OBJECT',
    'policy=bc_transformer_policy',
    'lifelong=base',
    'seed=10000',
    'folder=/Users/evanmanolis/LIBERO/libero/datasets',
    'bddl_folder=/Users/evanmanolis/Desktop/LIBERO_FORK/libero/libero/bddl_files',
    'init_states_folder=/Users/evanmanolis/Desktop/LIBERO_FORK/libero/libero/init_files',
    'device=cpu',
    'train.n_epochs=0',
    'eval.eval=false',
    'train.num_workers=1',
    'eval.num_workers=1',
]

res1 = run_cmd(cmd_workers1, env={'NUMBA_DISABLE_JIT': '1'}, expect_success=False)
assert res1.returncode != 0, 'Expected failure did not occur.'
assert 'h5py objects cannot be pickled' in (res1.stdout + res1.stderr)
print('EXPECTED BLOCKER #2 CONFIRMED')



$ /opt/homebrew/Caskroom/miniforge/base/envs/libero/bin/python3.8 libero/lifelong/main.py benchmark_name=LIBERO_OBJECT policy=bc_transformer_policy lifelong=base seed=10000 folder=/Users/evanmanolis/LIBERO/libero/datasets bddl_folder=/Users/evanmanolis/Desktop/LIBERO_FORK/libero/libero/bddl_files init_states_folder=/Users/evanmanolis/Desktop/LIBERO_FORK/libero/libero/init_files device=cpu train.n_epochs=0 eval.eval=false train.num_workers=1 eval.num_workers=1
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
{ 'bddl_folder': '/Users/evanmanolis/Desktop/LIBERO_FORK/libero/libero/bddl_files',
  'benchmark_name': 'LIBERO_OBJECT',
  'data': { 'action_scale': 1.0,
            'affine_translate': 4,
            'data_moda

In [32]:
# Final summary cell: quick pass/fail checklist.

print('Smoke checklist complete:')
print('1) interpreter fixed            -> PASS')
print('2) core imports                -> PASS')
print('3) robosuite with workaround   -> PASS')
print('4) dataset files exist         -> PASS')
print('5) all datasets load           -> PASS')
print('6) one-batch model loss        -> PASS')
print('7) known blockers reproduced   -> PASS (expected failures)')


Smoke checklist complete:
1) interpreter fixed            -> PASS
2) core imports                -> PASS
3) robosuite with workaround   -> PASS
4) dataset files exist         -> PASS
5) all datasets load           -> PASS
6) one-batch model loss        -> PASS
7) known blockers reproduced   -> PASS (expected failures)
